# SELMA — Community Training Notebook
## Fine-tune Llama 3.1 8B on Google Colab or Kaggle (Free Tier)

This notebook trains the **8B parameter community version** of SELMA on a single free GPU.

**Hardware requirements:**
- Google Colab free: T4 16GB — works (~6–8 hours)
- Google Colab Pro: A100 40GB — faster (~1.5 hours)
- Kaggle free: P100 16GB — works (~7–9 hours)

**Before you start:**
1. Accept the Llama 3.1 license at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
2. Get your HuggingFace token from https://huggingface.co/settings/tokens
3. In Colab: Runtime → Change runtime type → GPU


In [ ]:
# Verify GPU is available
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NOT FOUND — enable GPU in Runtime settings')

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate peft bitsandbytes trl datasets \
    lxml beautifulsoup4 requests pyyaml tqdm jsonlines huggingface-hub sentencepiece

# flash-attn is optional — skip if it fails (falls back to sdpa)
!pip install -q flash-attn --no-build-isolation 2>/dev/null || print('flash-attn skipped, using sdpa')

In [ ]:
# Clone the SELMA repo
!git clone https://codeberg.org/Ronin48/SELMA.git
%cd SELMA

In [ ]:
# Authenticate with HuggingFace (required for Llama 3.1)
from huggingface_hub import login
import getpass

hf_token = getpass.getpass('Enter your HuggingFace token (hf_...): ')
login(token=hf_token)

## Step 1: Collect Training Data

Choose which jurisdiction to train. Each state model is independent — you can train just one and submit the adapter as a contribution.

**Recommended for first-time contributors:** Georgia (federal + Georgia statutes, ~2–3 hours of data collection)

In [ ]:
JURISDICTION = 'georgia'  # change to your target state

# Download federal statutes (baseline for all models)
!python scripts/data_collection/fetch_federal_statutes.py

# Download state statutes
!python scripts/data_collection/fetch_state_statutes.py --jurisdiction {JURISDICTION}

# Download HuggingFace legal datasets
!python scripts/data_collection/fetch_legal_datasets.py

In [ ]:
# Generate synthetic training examples
# Set your Anthropic API key for synthetic data generation (optional but recommended)
import os
os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Anthropic API key (optional, press Enter to skip): ') or ''

if os.environ.get('ANTHROPIC_API_KEY'):
    !python scripts/data_collection/generate_synthetic_ai.py
else:
    !python scripts/data_collection/generate_synthetic.py
    print('Using rule-based synthetic generation (no API key needed)')

In [ ]:
# Prepare the dataset
!python scripts/training/prepare_dataset.py

import json
with open('data/processed/dataset_stats.json') as f:
    stats = json.load(f)
print('Dataset ready:')
for k, v in stats.items():
    print(f'  {k}: {v:,}')

## Step 2: Train

Uses the 8B config with settings tuned for free-tier GPUs. Reduce `batch_size` below if you get CUDA out-of-memory errors.

In [ ]:
# Patch batch size down if on T4/P100 (16GB)
import subprocess
gpu_mem = subprocess.run(
    ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
    capture_output=True, text=True
).stdout.strip()

vram_gb = int(gpu_mem) // 1024 if gpu_mem.isdigit() else 16
batch_size = 1 if vram_gb < 20 else 2
print(f'VRAM: ~{vram_gb}GB → using batch_size={batch_size}')

# Patch the config
import yaml
with open('configs/training_config_8b.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['training']['per_device_train_batch_size'] = batch_size
with open('configs/training_config_8b.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

In [ ]:
!python scripts/training/train_qlora.py --config configs/training_config_8b.yaml

## Step 3: Save and Share Your Adapter

The trained LoRA adapter is small (~500MB) and can be uploaded to HuggingFace for others to use or for inclusion in the main SELMA release.

In [ ]:
# Upload adapter to HuggingFace Hub
from huggingface_hub import HfApi
import os

hf_username = input('Your HuggingFace username: ')
repo_name = f'selma-8b-{JURISDICTION}-adapter'
repo_id = f'{hf_username}/{repo_name}'

api = HfApi()
api.create_repo(repo_id=repo_id, repo_type='model', exist_ok=True)
api.upload_folder(
    folder_path='output/selma-8b-qlora/final',
    repo_id=repo_id,
    repo_type='model',
)
print(f'\nAdapter uploaded to: https://huggingface.co/{repo_id}')
print('Open a pull request at https://codeberg.org/Ronin48/SELMA to contribute it!')

In [ ]:
# Quick inference test before uploading
!python -m src.selma.model \
    --config configs/model_config.yaml \
    --input "A suspect broke into a residence and assaulted the homeowner. Police found the suspect inside with a knife." \
    --no-thinking